## Question: "How long do I need to integrate NIRCam to reach a certain depth/SNR?"

### Some NIRCam Docs
- [Readout Patterns](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-detector-overview/nircam-detector-readout-patterns#gsc.tab=0)
- [Recommended Strategies](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-observing-strategies/nircam-imaging-recommended-strategies#gsc.tab=0)

In [1]:
# Pandeia reference data
import os
os.environ["pandeia_refdata"] = "./data/pandeia_data-2026.2-jwst"
os.environ["PSF_DIR"] = "./data/pandeia_psfs-2026.2-jwst"
os.environ["PYSYN_CDBS"] = "./data/grp/redcat/trds"

# Exposure time calculator
from pandeia.engine.calc_utils import build_default_calc
from pandeia.engine.perform_calculation import perform_calculation

# Telescope
telescope = 'jwst'
instrument = 'nircam'
mode = 'lw_imaging'
filter = 'f356w'
readout_pattern = "medium8" # "deep2", "deep8", "medium8", "shallow4", "bright2", "rapid"

# Science target: the depth and S/N we want to reach
target_depth = 30.0 # source brightness we want to detect (AB mag)
target_snr = 5.0 # required signal-to-noise (e.g. 5-sigma)

# Tuning exposure parameters (search space for the readout recipe)
min_groups = 5 # recommended, 1 for rapid
max_groups = 10 # 10 for all, except deep2 and deep8 = 20 max groups, rapid = max 2
min_nint = 1
max_nint = 10 # max allowed
nexp = 4
max_single_int = 1000 # seconds, avoid too-long single integrations. Longer for deep2,8. 1000s for all else

# Flux parameters
flux_unit = 'abmag' # flux unit

# 1) Set default calculation
calc = build_default_calc(telescope, instrument, mode)

# 2) Set instrument/filter and normalize the point source at the target depth
calc['configuration']['instrument']['filter'] = filter
calc['scene'][0]['shape']['geometry'] = 'point'
calc['scene'][0]['spectrum']['normalization'] = {
    'type': f'{telescope}',
    'bandpass': f'{instrument},{mode},{filter}',
    'norm_flux': target_depth,  # fixed: the magnitude we want to reach
    'norm_fluxunit': flux_unit
}

# 3) Detector settings (ngroup/nint are searched below; readout & nexp fixed)
calc['configuration']['detector']['readout_pattern'] = readout_pattern
calc['configuration']['detector']['nexp'] = nexp

# 4) Background level
calc['background_level'] = 'low'  # 'low', 'medium', 'high'

In [2]:
def solve_time_for_snr(calc, target_snr, min_groups, max_groups,
                       min_nint, max_nint, max_single_int):
    """
    Find the smallest-time (nint, ngroup) recipe that reaches S/N >= target_snr
    for the source already normalized in `calc`.

    We enumerate (nint, ngroup) recipes, order them by ~increasing total time
    (nint * ngroup is a good proxy), and evaluate until the target S/N is met.
    Because these are faint, unsaturated sources, S/N grows monotonically with
    time, so the first recipe that crosses the threshold is the (near-)minimum.
    Returns (rep, snr) at the winning recipe, or None if it can't be reached.
    """
    recipes = [(ni, ng) for ni in range(min_nint, max_nint + 1)
               for ng in range(min_groups, max_groups + 1)]
    recipes.sort(key=lambda r: (r[0] * r[1], r[0]))  # cheaper (shorter) first

    for nint, ngroup in recipes:
        calc['configuration']['detector']['ngroup'] = ngroup
        calc['configuration']['detector']['nint'] = nint
        rep = perform_calculation(calc)
        spec = rep['information']['exposure_specification']
        single_int = spec['exposure_time'] / nint  # per-integration time (s)

        # Skip recipes whose single integration is too long (cosmic rays)
        if single_int > max_single_int:
            continue

        snr = rep['scalar']['sn']
        tot = spec['total_exposure_time']
        print(f"[search] nint={nint}, ngroup={ngroup}, tot_exp={tot:.2f}s, "
              f"single_int={single_int:.2f}s, snr={snr:.2f}")

        if snr >= target_snr:
            return rep, snr

    return None  # target S/N not reachable within the search limits


# --- sqrt(t) sanity estimate (teaching aside) -------------------------------
# In the background/read-noise limit S/N grows as sqrt(time), so one reference
# exposure lets us predict the time needed before searching.
calc['configuration']['detector']['ngroup'] = max_groups
calc['configuration']['detector']['nint'] = 1
ref = perform_calculation(calc)
t_ref = ref['information']['exposure_specification']['total_exposure_time']
snr_ref = ref['scalar']['sn']
t_estimate = t_ref * (target_snr / snr_ref) ** 2
print(f"[estimate] one exposure = {t_ref:.0f}s gives S/N={snr_ref:.2f}; "
      f"sqrt(t) scaling => ~{t_estimate:.0f}s needed for S/N={target_snr}\n")

# --- solve for the actual exposure time -------------------------------------
result = solve_time_for_snr(calc, target_snr, min_groups, max_groups,
                            min_nint, max_nint, max_single_int)

print(f"\n=== {telescope}/{instrument}: time to reach m={target_depth} AB "
      f"at S/N={target_snr} ({filter}) ===")

if result is None:
    print(f"Target NOT reachable within the search limits "
          f"(max nint={max_nint}, max ngroup={max_groups}, nexp={nexp}).")
    print("Try: a deeper readout pattern, more dithers (nexp), or a wider filter.")
    
else:
    rep, snr = result
    spec = rep['information']['exposure_specification']
    tot = spec['total_exposure_time']
    print(f"Total exposure time: {tot:.2f} s  ({tot/3600:.2f} hours)")
    print(f"Recipe: readout={spec['readout_pattern']}, ngroup={spec['ngroup']}, "
          f"nint={spec['nint']}, nexp={spec['nexp']}")
    print(f"S/N at target depth: {snr:.2f}")

[estimate] one exposure = 4209s gives S/N=2.64; sqrt(t) scaling => ~15069s needed for S/N=5.0

[search] nint=1, ngroup=5, tot_exp=2061.46s, single_int=515.36s, snr=1.84
[search] nint=1, ngroup=6, tot_exp=2490.93s, single_int=622.73s, snr=2.04
[search] nint=1, ngroup=7, tot_exp=2920.40s, single_int=730.10s, snr=2.22
[search] nint=1, ngroup=8, tot_exp=3349.87s, single_int=837.47s, snr=2.37
[search] nint=1, ngroup=9, tot_exp=3779.34s, single_int=944.84s, snr=2.51
[search] nint=2, ngroup=5, tot_exp=4165.87s, single_int=520.73s, snr=2.61
[search] nint=2, ngroup=6, tot_exp=5024.81s, single_int=628.10s, snr=2.89
[search] nint=2, ngroup=7, tot_exp=5883.75s, single_int=735.47s, snr=3.14
[search] nint=3, ngroup=5, tot_exp=6270.27s, single_int=522.52s, snr=3.19
[search] nint=2, ngroup=8, tot_exp=6742.69s, single_int=842.84s, snr=3.36
[search] nint=2, ngroup=9, tot_exp=7601.63s, single_int=950.20s, snr=3.55
[search] nint=3, ngroup=6, tot_exp=7558.69s, single_int=629.89s, snr=3.54
[search] nint=4, 